In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#load datasets
customers = pd.read_csv("datasets/olist_customers_dataset.csv")
orders = pd.read_csv("datasets/olist_orders_dataset.csv")
items = pd.read_csv("datasets/olist_order_items_dataset.csv")
products = pd.read_csv("datasets/olist_products_dataset.csv")

In [ ]:
orders.head() #check the first few rows of the orders dataset

In [ ]:
orders['order_status'].value_counts() #check the distribution of order statuses, we will focus on delivered and canceled orders for return analysis

In [ ]:
orders = orders[orders['order_status'].isin(['delivered', 'canceled'])] #cleaning data, only keeping delivered and canceled orders

In [ ]:
#merge datasets to create a comprehensive dataframe for analysis
df = orders.merge(customers, on='customer_id') \
           .merge(items, on='order_id') \
           .merge(products, on='product_id')

In [ ]:
df.duplicated().sum() #check for duplicates, we should have none after merging

In [ ]:
df['is_return'] = df['order_status'] == 'canceled' #create a binary column (true/false) to indicate if an order was returned (canceled)
#true when order_status is 'canceled', false when 'delivered'
#dataset is now ready for analysis, we can explore return rates by product category, customer demographics, etc.

In [ ]:
df.head() #check the first few rows of the merged dataframe to confirm everything looks correct

In [ ]:
df['is_return'].value_counts() #check the distribution of returns vs non-returns.

In [ ]:
# RETURN RATE

return_rate = df['is_return'].mean() #true values are treated as 1, false as 0, so mean gives us the percentage of returns
print("Return Rate:", return_rate*100, "%")

In [ ]:
# Return rate counts
return_counts = df['is_return'].value_counts()
labels = ['Delivered', 'Returned']
colors = ['green', 'red']

plt.figure(figsize=(6,6))
plt.pie(return_counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
plt.title('Overall Return Rate')
plt.show()

# seaborn does not support pie chart

In [ ]:
# PRODUCT RETURN PATTERN

# Filter only returned orders
returns = df[df['is_return']] #Take all rows of df where is_return is True. This gives us a new dataframe called returns that only contains the returned orders.

# Count returns per product, sort by number of returns, highest first
product_returns = returns.groupby('product_id').size().sort_values(ascending=False)

# Top 10 returned products
top_products = product_returns.head(10)
print(top_products)

In [ ]:
top_products = returns['product_id'].value_counts().head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=top_products.values, y=top_products.index)

plt.title("Top 10 Returned Products (by ID)")
plt.xlabel("Number of Returns")
plt.ylabel("Product ID")

plt.show()

In [ ]:
# CATEGORY RISK

# Group by category: calculate return rate per category
category_risk = df.groupby('product_category_name')['is_return'].mean().sort_values(ascending=False)

# Top 10 risky categories
top_categories = category_risk.head(10)
print(top_categories)

In [ ]:
# Convert to percentage
top_categories_percent = top_categories * 100

plt.figure(figsize=(10,5))

sns.barplot(
    x=top_categories_percent.values,
    y=top_categories_percent.index,
    color='orange'
)

plt.xlabel('Return Rate (%)')
plt.title('Top 10 Risky Categories')

# Add labels
# top_categories_percent.values → all your return rates (numbers)
# enumerate() → gives:
# i = position (0, 1, 2, …) → y-axis position
# v = value (like 12.5, 8.3, etc.)
for i, v in enumerate(top_categories_percent.values):
    plt.text(v + 0.1, i, f'{v:.1f}%', va='center')

plt.show()